# NYC TLC Trip Data - Exploration & Profiling

EDA kept **separate from the pipeline code** (per the assignment). The pipeline
modules (`src/ingest.py`, `src/validate.py`, `src/model.py`, `src/metrics.py`)
must stay runnable without a notebook; this notebook exists to *show the work*
behind the business rules.

**What this notebook establishes (Class 6 - Profile & validate):**

1. Shape, dtypes and null profile of the raw monthly parquet files.
2. Distributions of the fields the KPI depends on (duration, fare, distance, passenger count).
3. The specific evidence behind each validation rule and judgment call in `src/validate.py`.
4. A look at the rejection reasons actually produced, so the ~3% reject rate is explainable.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
REJECTED = PROJECT_ROOT / "data" / "rejected"

# Make the pipeline modules importable so the notebook reuses their rules and
# profiling function rather than re-implementing them (single source of truth).
sys.path.insert(0, str(PROJECT_ROOT))
from src import validate  # noqa: E402

MONTH = "2026-01"
RAW_FILES = sorted(RAW.glob("*.parquet"))
print("Project root:", PROJECT_ROOT)
print("Raw parquet files:", [p.name for p in RAW_FILES])
print("Raw zone lookup present:", (RAW / "taxi_zone_lookup.csv").exists())
print("Weather files:", [p.name for p in sorted(RAW.glob("weather_*.csv"))])

Project root: /Users/akshatsipany/SST/fde/nyc-tlc-assignment-2
Raw parquet files: ['yellow_tripdata_2026-01.parquet', 'yellow_tripdata_2026-02.parquet', 'yellow_tripdata_2026-03.parquet']
Raw zone lookup present: True
Weather files: ['weather_2026-01.csv', 'weather_2026-02.csv', 'weather_2026-03.csv']


## 1. Raw inventory: are all three months present and plausible?

Ingest already ran completeness checks; this confirms it independently by reading
only parquet **metadata** (row counts) plus the two timestamp columns.

In [2]:
inventory = []
for path in RAW_FILES:
    pf = pq.ParquetFile(path)
    ts = pf.read(columns=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
    pickups = ts.column("tpep_pickup_datetime").to_pandas()
    dropoffs = ts.column("tpep_dropoff_datetime").to_pandas()
    inventory.append({
        "file": path.name,
        "rows": pf.metadata.num_rows,
        "cols": len(pf.schema_arrow.names),
        "min_pickup": pickups.min(),
        "max_pickup": pickups.max(),
        "min_dropoff": dropoffs.min(),
        "max_dropoff": dropoffs.max(),
    })

pd.DataFrame(inventory)

,file,rows,cols,min_pickup,max_pickup,min_dropoff,max_dropoff
0,yellow_tripdata_2026-01.parquet,3724889,20,2025-12-31 23:57:29,2026-02-01 00:45:01,2025-12-31 23:57:32,2026-02-01 23:35:31
1,yellow_tripdata_2026-02.parquet,3399866,20,2026-01-31 23:31:23,2026-03-01 00:51:48,2026-01-31 23:36:52,2026-03-01 23:11:29
2,yellow_tripdata_2026-03.parquet,3952451,20,2008-12-31 23:03:20,2026-04-01 00:06:25,2009-01-01 00:07:36,2026-04-02 16:03:47


**Observations worth recording:**

* Each month has ~3.4-4.0M trips - comfortably inside the 500k-6M band `ingest.py` enforces.
* Pickup timestamps spill slightly past month boundaries (e.g. late on the 31st into the 1st). This is **expected** TLC behavior, not corruption, which is why the month rule in `validate.py` pads the window by a day.
* The March file's `min_pickup` is **2008** - at least one badly mis-stamped record. It is far outside the padded month window, so the `pickup_outside_month` rule catches it. This is the concrete evidence for that rule.

## 2. Load one month and profile it

Working on a single month keeps the notebook quick and readable. The same code
runs per month in the pipeline; here it is interactive EDA.

In [3]:
raw = pd.read_parquet(RAW / f"yellow_tripdata_{MONTH}.parquet")
print("shape:", raw.shape)
raw.head()

shape: (3724889, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [4]:
# Reuse the pipeline's own profiling function so the notebook and the pipeline
# never disagree about what "profiled" means.
summary = validate.profile(raw)
summary

2026-09-25 12:04:13,401 | INFO | validate | [profile] shape: 3,724,889 rows x 20 cols


2026-09-25 12:04:13,403 | INFO | validate | [profile] dtypes:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64


2026-09-25 12:04:13,510 | INFO | validate | [profile] null counts (non-zero only):
passenger_count         1088058
RatecodeID              1088058
store_and_fwd_flag      1088058
congestion_surcharge    1088058
Airport_fee             1088058


2026-09-25 12:04:13,619 | INFO | validate | [profile] duration_min quantiles:
0.00     -11.70
0.01       0.00
0.50      13.37
0.99      71.43
1.00    7507.90


2026-09-25 12:04:13,688 | INFO | validate | [profile] fare_amount quantiles:
0.00   -2555.2
0.01      -3.0
0.50      15.6
0.99      81.4
1.00    2555.2


2026-09-25 12:04:13,763 | INFO | validate | [profile] trip_distance quantiles:
0.00         0.00
0.01         0.00
0.50         1.81
0.99        19.54
1.00    269097.48


2026-09-25 12:04:13,809 | INFO | validate | [profile] passenger_count quantiles:
0.00    0.0
0.01    1.0
0.50    1.0
0.99    4.0
1.00    9.0


2026-09-25 12:04:16,131 | INFO | validate | [profile] duplicates: 0 full-row, 35723 on (pickup, dropoff, PU, DO)


2026-09-25 12:04:16,138 | INFO | validate | [profile] pickup range: 2025-12-31 23:57:29 .. 2026-02-01 00:45:01


{'rows': 3724889,
 'cols': 20,
 'nulls': {'passenger_count': 1088058,
  'RatecodeID': 1088058,
  'store_and_fwd_flag': 1088058,
  'congestion_surcharge': 1088058,
  'Airport_fee': 1088058},
 'dup_full': 0,
 'dup_on_key': 35723}

### Null profile: where is data actually missing?

This is the single most consequential finding in the whole dataset.

In [5]:
nulls = raw.isna().sum().sort_values(ascending=False)
null_pct = (nulls / len(raw) * 100).round(2)
pd.DataFrame({"null_count": nulls, "pct_of_rows": null_pct}).query("null_count > 0")

,null_count,pct_of_rows
Airport_fee,1088058,29.21
passenger_count,1088058,29.21
congestion_surcharge,1088058,29.21
RatecodeID,1088058,29.21
store_and_fwd_flag,1088058,29.21


## 3. The 29% null `passenger_count` block - the key judgment call

`passenger_count` is null for ~29% of rows, and the nulls line up **exactly** with
nulls in `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee`
and `payment_type == 0`. That exact alignment is not random missingness - it is a
contiguous block of records from a partial upstream feed.

**The question:** does the assignment's rule `passenger_count between 1 and 6` mean
these rows should be rejected?

Rejecting them would throw away **29% of all trips** - and none of the five metrics
use passenger count at all. Let's check whether these rows look like real trips.

In [6]:
missing_pc = raw["passenger_count"].isna()
block = raw.loc[missing_pc]

idx = block.index
print(f"rows with null passenger_count: {len(block):,} of {len(raw):,} ({missing_pc.mean():.1%})")
print(f"index range: {idx.min()} .. {idx.max()}")
print(f"is it one contiguous block? {(idx.max() - idx.min() + 1) == len(idx)}")
print("\nVendorID mix inside the block:", block["VendorID"].value_counts().to_dict())
print("payment_type inside the block:", block["payment_type"].value_counts().to_dict())
print(f"pickup range inside block: {block['tpep_pickup_datetime'].min()} .. {block['tpep_pickup_datetime'].max()}")

rows with null passenger_count: 1,088,058 of 3,724,889 (29.2%)
index range: 2636831 .. 3724888
is it one contiguous block? True

VendorID mix inside the block: {2: 944635, 1: 139406, 6: 4017}
payment_type inside the block: {0: 1088058}
pickup range inside block: 2026-01-01 00:00:00 .. 2026-01-31 23:59:59


In [7]:
all_dur = validate.duration_minutes(raw)
block_dur = validate.duration_minutes(block)
comparison = pd.DataFrame({
    "null_passenger_count block": {
        "median_duration_min": round(block_dur.median(), 1),
        "p99_duration_min": round(block_dur.quantile(0.99), 1),
        "median_fare": round(block["fare_amount"].median(), 2),
        "invalid_durations": int(((block_dur < validate.MIN_DURATION_MIN) | (block_dur > validate.MAX_DURATION_MIN)).sum()),
    },
    "all rows": {
        "median_duration_min": round(all_dur.median(), 1),
        "p99_duration_min": round(all_dur.quantile(0.99), 1),
        "median_fare": round(raw["fare_amount"].median(), 2),
        "invalid_durations": int(((all_dur < validate.MIN_DURATION_MIN) | (all_dur > validate.MAX_DURATION_MIN)).sum()),
    },
})
comparison

,null_passenger_count block,all rows
median_duration_min,16.00,13.4
p99_duration_min,51.10,71.4
median_fare,22.21,15.6
invalid_durations,3123.00,85290.0


The null block's median duration and median fare are in line with the rest of the
data, and only a tiny fraction of its rows fail the duration rules outright. These
are **real trips with missing metadata**, not garbage.

> **Decision:** a *missing* passenger count does **not** invalidate a trip; a *populated*
> value outside 1-6 does. No metric depends on passenger count, so rejecting 29% of
> the data would cost accuracy and buy nothing.

## 4. Duration distribution - why the lower bound is 1 minute, not 0

The assignment suggests `duration > 0`. Let's look at what sits between 0 and 1 minute.

In [8]:
dur = validate.duration_minutes(raw)

buckets = pd.DataFrame({
    "duration (minutes)": ["== 0", "(0, 0.1)", "[0.1, 0.5)", "[0.5, 1)", "[1, 2)", "[2, 5)", "[5, 240)", ">= 240"],
    "rows": [
        int((dur == 0).sum()),
        int(((dur > 0) & (dur < 0.1)).sum()),
        int(((dur >= 0.1) & (dur < 0.5)).sum()),
        int(((dur >= 0.5) & (dur < 1)).sum()),
        int(((dur >= 1) & (dur < 2)).sum()),
        int(((dur >= 2) & (dur < 5)).sum()),
        int(((dur >= 5) & (dur < 240)).sum()),
        int((dur >= 240).sum()),
    ],
})
buckets

,duration (minutes),rows
0,== 0,45069
1,"(0, 0.1)",8292
2,"[0.1, 0.5)",23387
3,"[0.5, 1)",6998
4,"[1, 2)",22538
5,"[2, 5)",289107
6,"[5, 240)",3327954
7,>= 240,1543


In [9]:
# Are the zero/near-zero durations real trips, or timestamp artifacts?
instant = raw.loc[dur < 1]
print(f"rows under 1 minute: {len(instant):,}")
print(f"  of those, trip_distance == 0: {int((instant['trip_distance'] == 0).sum()):,}")
print(f"  of those, trip_distance  > 0: {int((instant['trip_distance'] > 0).sum()):,}")
print(f"  median distance among them: {instant['trip_distance'].median():.3f} mi")
print(f"  median fare among them:     ${instant['fare_amount'].median():.2f}")

zero = raw.loc[dur == 0]
print(f"\nrows where pickup == dropoff exactly: {len(zero):,}")
print(f"  ...that still report trip_distance > 0: {int((zero['trip_distance'] > 0).sum()):,}")
print("  -> a trip cannot cover ground in zero seconds; these are timestamp artifacts.")

rows under 1 minute: 83,747
  of those, trip_distance == 0: 27,160
  of those, trip_distance  > 0: 56,587
  median distance among them: 0.500 mi
  median fare among them:     $12.10

rows where pickup == dropoff exactly: 45,069
  ...that still report trip_distance > 0: 43,905
  -> a trip cannot cover ground in zero seconds; these are timestamp artifacts.


> **Decision:** `MIN_DURATION_MIN = 1.0` rather than the assignment's `> 0`. Tens of
> thousands of rows have pickup == dropoff to the second *while reporting real
> distance*, and tens of thousands more last under a minute. These would corrupt the
> duration KPI. The threshold is a single named constant in `validate.py`.

## 5. Outliers: fare and distance

Two outlier classes matter, and they are treated differently on purpose.

In [10]:
for col in ["fare_amount", "trip_distance"]:
    s = raw[col]
    print(f"--- {col}")
    print(s.quantile([0, 0.001, 0.01, 0.5, 0.99, 0.999, 1]).round(2).to_string())
    print(f"    negatives: {int((s < 0).sum()):,} | zeros: {int((s == 0).sum()):,} | over 100: {int((s > 100).sum()):,}\n")

print("Negative fares are refunds/disputes -> rejected (assignment rule: fare >= 0).")
print("Fare == 0 and distance == 0 are KEPT: the zero-fare/zero-distance rate is metric 5,")
print("so those rows are the evidence, not the error.")

--- fare_amount
0.000   -2555.2
0.001     -70.0
0.010      -3.0
0.500      15.6
0.990      81.4
0.999     145.8
1.000    2555.2
    negatives: 39,463 | zeros: 2,082 | over 100: 12,593

--- trip_distance
0.000         0.00
0.001         0.00
0.010         0.00
0.500         1.81
0.990        19.54
0.999        29.79
1.000    269097.48
    negatives: 0 | zeros: 125,738 | over 100: 162

Negative fares are refunds/disputes -> rejected (assignment rule: fare >= 0).
Fare == 0 and distance == 0 are KEPT: the zero-fare/zero-distance rate is metric 5,
so those rows are the evidence, not the error.


In [11]:
# Distance outliers: are >100 mile trips plausible for a NYC yellow cab?
far = raw.loc[raw["trip_distance"] > 100].sort_values("trip_distance", ascending=False)
print(f"trips over 100 miles: {len(far):,}")
far[["tpep_pickup_datetime", "PULocationID", "DOLocationID", "trip_distance", "fare_amount", "tpep_dropoff_datetime"]].head(10)

trips over 100 miles: 162


,tpep_pickup_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,tpep_dropoff_datetime
3524923,2026-01-28 01:08:00,79,144,269097.48,12.19,2026-01-28 01:16:00
3632367,2026-01-30 13:00:00,42,130,237567.19,72.99,2026-01-30 13:38:00
3285750,2026-01-21 16:31:00,234,107,236216.69,13.58,2026-01-21 16:40:00
3131676,2026-01-17 10:40:00,137,82,236183.06,22.75,2026-01-17 10:57:00
2704619,2026-01-02 21:31:00,249,41,221544.91,31.21,2026-01-02 21:55:00
2688274,2026-01-02 07:07:00,231,161,221436.73,27.61,2026-01-02 07:18:00
3131406,2026-01-17 10:39:00,13,162,196490.48,33.59,2026-01-17 10:56:00
3571687,2026-01-29 07:51:00,33,246,192081.68,60.20,2026-01-29 08:26:00
2820951,2026-01-07 17:51:00,164,186,184908.33,12.18,2026-01-07 17:56:00
3499856,2026-01-27 14:46:00,230,74,183363.75,71.01,2026-01-27 16:10:00


A reported max distance in the hundreds of thousands of miles is a decimal-placement
error, not a trip. Airport runs (JFK <-> Manhattan) are the genuinely long ones at
~30-35 miles.

> **Decision:** `MAX_TRIP_DISTANCE_MI = 100.0`. Only ~162 rows/month, but they would
> badly distort revenue-per-mile (metric 3), so this is about metric validity.

## 6. Duplicates

Zero full-row duplicates across all three months, but tens of thousands of rows share
the (pickup, dropoff, PU, DO) key. Are those real duplicated trips?

In [12]:
key = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"]
print("full-row duplicates:", int(raw.duplicated().sum()))
print("duplicates on (pickup, dropoff, PU, DO):", int(raw.duplicated(subset=key).sum()))

sample = raw.loc[raw.duplicated(subset=key, keep=False)].sort_values(key).head(8)
sample[key + ["trip_distance", "fare_amount", "total_amount", "VendorID"]]

full-row duplicates: 0


duplicates on (pickup, dropoff, PU, DO): 35723


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount,VendorID
980,2025-12-31 23:57:29,2025-12-31 23:57:32,132,264,0.01,-70.0,-71.50,2
981,2025-12-31 23:57:29,2025-12-31 23:57:32,132,264,0.01,70.0,71.50,2
5065,2026-01-01 00:01:19,2026-01-01 00:04:08,48,163,0.05,-4.4,-10.15,2
5066,2026-01-01 00:01:19,2026-01-01 00:04:08,48,163,0.05,4.4,10.15,2
428,2026-01-01 00:01:24,2026-01-01 00:02:02,162,162,0.03,-3.0,-8.75,2
429,2026-01-01 00:01:24,2026-01-01 00:02:02,162,162,0.03,3.0,8.75,2
1046,2026-01-01 00:01:52,2026-01-01 00:04:22,234,234,0.14,-3.7,-9.45,2
1047,2026-01-01 00:01:52,2026-01-01 00:04:22,234,234,0.14,3.7,9.45,2


Note the same key appears with **different distances and fares** - these are genuinely
different trips that happened to share a second-level timestamp and zone pair, not
duplicate records.

> **Decision:** do not deduplicate. Rows are flagged with `is_duplicate_key` for the
> data-quality view, but kept. Silently deleting on this key would risk removing real trips.

## 7. Zone lookup coverage

The rule `PULocationID / DOLocationID exist in the zone lookup` is required by the
assignment, so let's confirm how often it actually fires.

In [13]:
zones = pd.read_csv(RAW / "taxi_zone_lookup.csv")
valid_ids = set(zones["LocationID"])
print(f"lookup rows: {len(zones)} | unique LocationIDs: {zones['LocationID'].nunique()}")
print(f"pickup IDs not in lookup:  {int((~raw['PULocationID'].isin(valid_ids)).sum()):,}")
print(f"dropoff IDs not in lookup: {int((~raw['DOLocationID'].isin(valid_ids)).sum()):,}")
print("\nRule retained as a guard even though it currently rejects nothing:")
print("a future TLC schema change or a different borough's IDs would break the join silently.")
zones.head()

lookup rows: 265 | unique LocationIDs: 265
pickup IDs not in lookup:  0
dropoff IDs not in lookup: 0

Rule retained as a guard even though it currently rejects nothing:
a future TLC schema change or a different borough's IDs would break the join silently.


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


## 8. Weather data: join key and timezone

Weather is joined to trips on the **local New York pickup hour**, so both sides must
be in the same timezone for metric 4 to be trustworthy.

In [14]:
weather = pd.read_csv(RAW / f"weather_{MONTH}.csv", parse_dates=["time"])
print("weather shape:", weather.shape)
print(weather.head().to_string(index=False))
print("\nfirst hour:", weather['time'].min(), "| last hour:", weather['time'].max())
print(f"hours in {MONTH}: {len(weather)} (expected {pd.Period(MONTH).days_in_month * 24})")
print(f"rainy hours (precipitation > 0): {int((weather['precipitation'] > 0).sum()):,}")
print(f"null measurements: {int(weather[['temperature_2m', 'precipitation']].isna().sum().sum())}")
print("\nTimestamps are LOCAL America/New_York (matching TLC pickup time) -")
print("so the join in model.py is a direct hour lookup with no conversion.")

weather shape: (744, 3)
               time  temperature_2m  precipitation
2026-01-01 00:00:00            -1.0            0.0
2026-01-01 01:00:00            -0.8            0.1
2026-01-01 02:00:00            -0.4            0.1
2026-01-01 03:00:00            -0.2            0.0
2026-01-01 04:00:00             0.4            0.0

first hour: 2026-01-01 00:00:00 | last hour: 2026-01-31 23:00:00
hours in 2026-01: 744 (expected 744)
rainy hours (precipitation > 0): 83
null measurements: 0

Timestamps are LOCAL America/New_York (matching TLC pickup time) -
so the join in model.py is a direct hour lookup with no conversion.


## 9. Validation results: what actually got rejected?

Now inspect the artifacts `validate.py` already wrote. This reads the pipeline's own
outputs rather than re-implementing the rules.

In [15]:
clean = pd.read_parquet(PROCESSED / f"{MONTH}_clean.parquet")
# The rejected artifact is CSV, so timestamps come back as strings:
# parse them explicitly before any duration math.
rejected = pd.read_csv(REJECTED / f"{MONTH}_rejected.csv",
                         parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"])

total = len(clean) + len(rejected)
print(f"raw rows:   {total:,}")
print(f"clean:      {len(clean):,} ({len(clean)/total:.2%})")
print(f"rejected:   {len(rejected):,} ({len(rejected)/total:.2%})")
assert len(clean) + len(rejected) == len(raw), "clean + rejected must equal raw rows"
print("\nOK: clean + rejected reconciles exactly to the raw row count (nothing silently dropped).")

raw rows:   3,724,889
clean:      3,590,046 (96.38%)
rejected:   134,843 (3.62%)

OK: clean + rejected reconciles exactly to the raw row count (nothing silently dropped).


In [16]:
reason_counts = (rejected["rejection_reason"].str.split(";").explode().value_counts())
reason_frame = reason_counts.to_frame("rows")
reason_frame["pct_of_raw"] = (reason_frame["rows"] / total * 100).round(2)
reason_frame["pct_of_rejected"] = (reason_frame["rows"] / len(rejected) * 100).round(2)
reason_frame

,rows,pct_of_raw,pct_of_rejected
rejection_reason,,,
invalid_duration,85290,2.29,63.25
negative_fare,39463,1.06,29.27
invalid_passenger_count,14794,0.40,10.97
implausible_distance,162,0.00,0.12
pickup_outside_month,1,0.00,0.00


In [17]:
# Rows can fail several rules; every reason is recorded (semicolon-joined).
multi = rejected.loc[rejected["rejection_reason"].str.contains(";")]
print(f"rows failing more than one rule: {len(multi):,} ({len(multi)/len(rejected):.1%} of rejects)")
multi["rejection_reason"].value_counts().head(10)

rows failing more than one rule: 4,865 (3.6% of rejects)


rejection_reason
invalid_duration;negative_fare                            4327
invalid_duration;invalid_passenger_count                   506
negative_fare;invalid_passenger_count                       20
negative_fare;implausible_distance                           6
invalid_duration;implausible_distance                        3
invalid_duration;negative_fare;invalid_passenger_count       2
implausible_distance;invalid_passenger_count                 1
Name: count, dtype: int64

### Spot-check: are the rejects genuinely unusable?

The strongest evidence that validation is calibrated correctly is that the rejected
rows are indefensible while the kept ones are ordinary trips.

In [18]:
def summarize(frame, label, duration_min):
    return {
        label: {
            "rows": len(frame),
            "median_duration_min": round(duration_min.median(), 1),
            "median_distance_mi": round(frame["trip_distance"].median(), 2),
            "median_fare": round(frame["fare_amount"].median(), 2),
        }
    }

clean_dur = validate.duration_minutes(clean)
rej_dur = validate.duration_minutes(rejected)

pd.DataFrame([
    summarize(clean, "CLEAN", clean_dur),
    summarize(rejected, "REJECTED", rej_dur),
])

,CLEAN,REJECTED
0,"{'rows': 3590046, 'median_duration_min': 13.6,...",NaN
1,NaN,"{'rows': 134843, 'median_duration_min': 0.2, '..."


## 10. Data quality of the KEPT data (flags carried into the model)

`validate.py` attaches two flags to clean rows rather than dropping on them.

In [19]:
print("passenger_count_missing (kept, flagged):", f"{int(clean['passenger_count_missing'].sum()):,}")
print("is_duplicate_key (kept, flagged):     ", f"{int(clean['is_duplicate_key'].sum()):,}")
print("\nflagged rows are retained because no metric uses passenger_count, and shared")
print("keys carry differing fares/distances (real trips), so removal would lose signal.")
clean.head()

passenger_count_missing (kept, flagged): 1,084,785
is_duplicate_key (kept, flagged):      32,099

flagged rows are retained because no metric uses passenger_count, and shared
keys carry differing fares/distances (real trips), so removal would lose signal.


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,passenger_count_missing,is_duplicate_key
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00,False,False
1,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75,False,False
2,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75,False,False
3,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75,False,False
4,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75,False,False


## 11. Post-validation distribution sanity check

After cleaning, the duration distribution should look like real trips: no mass at zero,
a plausible median, and a long-but-sane right tail.

In [20]:
print("BEFORE validation:")
print(validate.duration_minutes(raw).quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).round(1).to_string())
print("\nAFTER validation:")
print(clean_dur.quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]).round(1).to_string())
print(f"\nmin duration before: {validate.duration_minutes(raw).min():.1f} min | after: {clean_dur.min():.1f} min")
print(f"max duration before: {validate.duration_minutes(raw).max():.1f} min | after: {clean_dur.max():.1f} min")

BEFORE validation:


0.00     -11.7
0.01       0.0
0.25       8.1
0.50      13.4
0.75      21.2
0.99      71.4
1.00    7507.9

AFTER validation:


0.00      1.0
0.01      2.4
0.25      8.4
0.50     13.6
0.75     21.4
0.99     71.1
1.00    239.0

min duration before: -11.7 min | after: 1.0 min
max duration before: 7507.9 min | after: 239.0 min


In [21]:
# Hour-of-day volume profile: the axis metrics 1 and 2 are built on.
hourly = clean.assign(hour=clean["tpep_pickup_datetime"].dt.hour).groupby("hour").size()
hourly.rename("trips").to_frame()

,trips
hour,
0,108169
1,75079
2,52057
3,37145
4,27450
5,31599
6,61472
7,109613
8,143141


Volume peaks in the evening rush and collapses after 3am, as expected for NYC yellow
cabs. This confirms `pickup_hour` is a sound grouping key for the zone x hour metrics.

## 12. Consolidated decision log

Every judgment call made in `validate.py`, with the evidence that justified it. These
must stay in sync with the README Assumptions section (traceability is graded).

| Rule / decision | Threshold | Evidence | Rationale |
|---|---|---|---|
| `invalid_duration` | 1 min <= d <= 240 min | 45,069 zero-second rows (43,905 with real distance) | Assignment said `>0`; sub-minute trips are timestamp artifacts that corrupt the KPI |
| `negative_fare` | fare >= 0 | ~20k-40k negative fares per month | Assignment rule; negatives are refunds/disputes |
| `implausible_distance` | <= 100 mi | max in the hundreds of thousands of miles; ~162 rows/month | Decimal-placement errors; would wreck revenue-per-mile |
| `unknown_pickup_zone` / `unknown_dropoff_zone` | ID in lookup | 0 violations observed | Assignment rule, kept as a guard against future schema changes |
| `invalid_passenger_count` | 1-6, **only when populated** | ~29% null in one contiguous upstream block; block durations/fares are normal | Missing metadata != invalid trip; no metric uses passenger count |
| `pickup_outside_month` | month +/- 1 day | 1-4 rows/month; a 2008 timestamp in the March file | Boundary spillover is real TLC behavior; 2008 is not |
| zero-fare / zero-distance rows | **KEPT** | thousands of zero fares; >120k zero distances per month | They are the subject of metric 5 |
| duplicate-key rows | **KEPT + flagged** | 0 full-row dupes; shared keys have differing fares | Shared second-level timestamps are not proof of duplication |

## 13. Implications for the modelling stage

Carried into `model.py` / `metrics.py`:

* ~3.6M clean rows per month, with `passenger_count_missing` and `is_duplicate_key`
  available as quality flags.
* Duration is trustworthy (1-240 min), so the median-per-zone-pair/hour benchmark
  behind metrics 2 and 4 is well-founded.
* `PULocationID` / `DOLocationID` always resolve in the lookup, so the zone joins are safe.
* Zero-distance and zero-fare rows are present in the clean set, so metric 5 is computable.
* Weather is a complete hourly local-time series, so the rainy/dry join in metric 4
  needs no timezone conversion or gap fallback.
* Known caveat: the ~29% partial-feed block has no passenger count or payment type, so
  any future metric touching those fields would need to handle the gap explicitly.